# Medical Insurance Cost Prediction using Multiple Linear Regression



## Problem Statement
An insurance company wants to estimate the medical insurance charges of customers based on their personal and health-related features. Develop a Multiple Linear Regression model to predict insurance charges.


## Step 0: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import urllib.request
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print("All libraries imported successfully!")

---

# Task 1: Data Understanding (2 Marks)

### Requirements:
1. Load the dataset using Pandas
2. Display the first five records
3. Identify:
   - Numerical features
   - Categorical features
   - Target variable

In [ ]:
# ============================================================
# 1.1 Load the dataset using Pandas
# ============================================================

# Dataset URL (public source - same data as Kaggle)
url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

# Download and load the dataset
urllib.request.urlretrieve(url, "insurance.csv")
df = pd.read_csv("insurance.csv")

print("Dataset loaded successfully!")
print(f"Shape: {df.shape} (rows, columns)")
print(f"\nColumn names: {list(df.columns)}")

In [ ]:
# ============================================================
# 1.2 Display the first five records
# ============================================================
print("First 5 records of the dataset:")
df.head()

In [ ]:
# ============================================================
# 1.3 Dataset Information
# ============================================================
print("Dataset Info:")
df.info()
print("\n" + "="*50)
print("Dataset Description (Numerical Columns):")
df.describe()

In [ ]:
# ============================================================
# 1.3 Identify Feature Types
# ============================================================

# Numerical features (excluding target)
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
if 'charges' in numerical_features:
    numerical_features.remove('charges')

# Categorical features
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

# Target variable
target_variable = 'charges'

print("=" * 50)
print("FEATURE IDENTIFICATION")
print("=" * 50)
print(f"Numerical Features   : {numerical_features}")
print(f"Categorical Features : {categorical_features}")
print(f"Target Variable      : {target_variable}")
print("=" * 50)

---

# Task 2: Data Preprocessing (2 Marks)

### Requirements:
1. Check for missing values
2. Encode categorical variables (sex, smoker, region)
3. Split dataset into 80% training and 20% testing

In [ ]:
# ============================================================
# 2.1 Check for Missing Values
# ============================================================
print("Missing Values in each column:")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")
if missing.sum() == 0:
    print("No missing values found in the dataset!")

In [ ]:
# ============================================================
# 2.2 Encode Categorical Variables
# ============================================================

# Create a copy for preprocessing
df_processed = df.copy()

print("Encoding categorical variables...")
print("-" * 40)

# Encode 'sex': female=0, male=1
le_sex = LabelEncoder()
df_processed['sex'] = le_sex.fit_transform(df_processed['sex'])
print(f"Sex encoding: {dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_)))}")

# Encode 'smoker': no=0, yes=1
le_smoker = LabelEncoder()
df_processed['smoker'] = le_smoker.fit_transform(df_processed['smoker'])
print(f"Smoker encoding: {dict(zip(le_smoker.classes_, le_smoker.transform(le_smoker.classes_)))}")

# Encode 'region' using One-Hot Encoding
region_dummies = pd.get_dummies(df_processed['region'], prefix='region')
df_processed = pd.concat([df_processed.drop('region', axis=1), region_dummies], axis=1)
print(f"Region encoded with One-Hot Encoding: {list(region_dummies.columns)}")

print("\nProcessed Dataset (first 5 rows):")
df_processed.head()

In [ ]:
# ============================================================
# 2.3 Split into Training (80%) and Testing (20%)
# ============================================================

# Features (X) and Target (y)
X = df_processed.drop('charges', axis=1)
y = df_processed['charges']

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Dataset Split Summary:")
print("-" * 40)
print(f"Training set size  : {X_train.shape[0]} samples ({X_train.shape[0]/len(df)*100:.0f}%)")
print(f"Testing set size   : {X_test.shape[0]} samples ({X_test.shape[0]/len(df)*100:.0f}%)")
print(f"Number of features : {X.shape[1]}")
print(f"Features used      : {list(X.columns)}")

---

# Task 3: Model Development (3 Marks)

### Requirements:
Build a Multiple Linear Regression model using:
- **Features:** Age, Sex, BMI, Children, Smoker, Region
- **Target Variable:** Charges

Train the model and predict charges for the test set.

In [ ]:
# ============================================================
# Build Multiple Linear Regression Model
# ============================================================

# Initialize the model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

print("Model trained successfully!")
print("=" * 50)
print("MODEL COEFFICIENTS")
print("=" * 50)
print(f"{'Feature':<25} {'Coefficient':>12}")
print("-" * 50)
for feature, coef in zip(X.columns, model.coef_):
    print(f"{feature:<25} {coef:>12.4f}")
print("-" * 50)
print(f"{'Intercept':<25} {model.intercept_:>12.4f}")
print("=" * 50)

In [ ]:
# ============================================================
# Predict Charges for Test Set
# ============================================================

y_pred = model.predict(X_test)

print("Predictions on Test Set (first 10 samples):")
print("-" * 60)
comparison = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10],
    'Difference': y_test.values[:10] - y_pred[:10]
})
comparison

---

# Task 4: Model Evaluation (2 Marks)

### Requirements:
Evaluate the model using:
- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- R² Score

Also create: Actual vs Predicted scatter plot

Write 2-3 observations based on the model's performance.

In [ ]:
# ============================================================
# 4.1 Calculate Evaluation Metrics
# ============================================================

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)

print("=" * 50)
print("MODEL EVALUATION METRICS")
print("=" * 50)
print(f"Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"Mean Squared Error (MSE)  : ${mse:,.2f}")
print(f"Root Mean Squared Error   : ${rmse:,.2f}")
print(f"R² Score                  : {r2:.4f}")
print("=" * 50)

In [ ]:
# ============================================================
# 4.2 Actual vs Predicted Scatter Plot
# ============================================================

plt.figure(figsize=(10, 7))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Charges ($)', fontsize=12)
plt.ylabel('Predicted Charges ($)', fontsize=12)
plt.title('Actual vs Predicted Insurance Charges\n(Multiple Linear Regression)', 
          fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print("Scatter plot saved as 'actual_vs_predicted.png'")

In [ ]:
# ============================================================
# 4.3 Feature Importance Visualization
# ============================================================

plt.figure(figsize=(10, 6))
feature_names = X.columns
feature_importance = np.abs(model.coef_)
colors = ['#e74c3c' if f == 'smoker' else '#3498db' for f in feature_names]
bars = plt.barh(feature_names, feature_importance, color=colors, edgecolor='white')
plt.xlabel('Absolute Coefficient Value', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Feature Importance in Insurance Cost Prediction\n(Absolute Coefficient Values)', 
          fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, feature_importance):
    plt.text(val + 200, bar.get_y() + bar.get_height()/2, 
             f'{val:,.0f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Feature importance chart saved as 'feature_importance.png'")

In [ ]:
# ============================================================
# 4.4 Residuals Distribution
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of charges
axes[0].hist(df['charges'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Charges ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Insurance Charges')
axes[0].grid(alpha=0.3)

# Distribution of residuals
residuals = y_test - y_pred
axes[1].hist(residuals, bins=30, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Residuals (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Distribution charts saved as 'distributions.png'")

### Observations (2-3 points):

1. **The model achieves an R² score of 0.7836**, meaning approximately 78.4% of the variance in insurance charges is explained by the features. This indicates a moderate fit — the model captures general trends but misses some patterns.

2. **Smoking status has by far the strongest impact on charges** (coefficient ≈ 23,651), indicating smokers pay significantly higher premiums. This aligns with real-world expectations where smoking is a major health risk factor.

3. **The scatter plot shows the model struggles with higher charges (>$30,000)**, where predictions tend to be less accurate. This suggests non-linear relationships or interaction effects (e.g., age + smoker) that linear regression cannot capture.

---

# Task 5: Conclusion (1 Mark)

Write a 100-150 word conclusion covering:
- Key findings
- Factors affecting insurance charges
- One limitation of Linear Regression for this problem

## Conclusion

This project developed a **Multiple Linear Regression model** to predict medical insurance charges using personal and health-related features. The model achieved an **R² score of 0.7836**, explaining approximately 78.4% of the variance in charges.

**Key findings** reveal that **smoking status** is the most dominant factor affecting insurance costs (coefficient ≈ 23,651), followed by **BMI** and **age**. Smokers pay significantly higher premiums, which aligns with real-world actuarial practices. The number of children and region also contribute but to a lesser extent.

**One major limitation** of Linear Regression for this problem is its assumption of linearity. Insurance charges often exhibit non-linear relationships and interaction effects (e.g., age combined with smoking status). The scatter plot reveals that the model underperforms for high-cost cases (>$30,000), suggesting that more advanced models like Polynomial Regression or Random Forest could yield better predictions for extreme values.